In [1]:
import pandas as pd

# Load sentiment scores (TSV)
scores_df = pd.read_csv("./datasetFolder/t4sa_text_sentiment.tsv", sep="\t", names=["TWID", "NEG", "NEU", "POS"], skiprows=1)

# Load tweet text (CSV)
tweets_df = pd.read_csv("./datasetFolder/raw_tweets_text.csv")

# Merge on 'id' from tweets_df and 'TWID' from scores_df
merged_df = pd.merge(tweets_df, scores_df, left_on="id", right_on="TWID")

# Function to assign label based on max score
def get_label(row):
    scores = {"NEG": row["NEG"], "NEU": row["NEU"], "POS": row["POS"]}
    return max(scores, key=scores.get)

# Map label to integers
label_map = {"NEG": 0, "POS": 1, "NEU": 2}
merged_df["label"] = merged_df.apply(get_label, axis=1).map(label_map)

# Keep useful columns
final_df = merged_df[["text", "label"]]

print(final_df.head())


                                                text  label
0  Josh Jenkins is looking forward to TAB Breeder...      1
1  RT @2pmthailfans: [Pic] Nichkhun from krjeong8...      2
2  RT @MianUsmanJaved: Congratulations Pakistan o...      1
3  RT @PEPalerts: This September, @YESmag is taki...      1
4  #Incredible #India #Atulya #Bharat - Land of S...      2


In [5]:
# ---------- PIE CHART for ORIGINAL distribution ----------
original_counts = final_df["label"].value_counts().sort_index()
labels = ['Negative', 'Positive', 'Neutral']
colors = ['#FF6B6B', '#6BCB77', '#4D96FF']

plt.figure(figsize=(6, 6))
plt.pie(original_counts, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title("Original Sentiment Distribution")
plt.savefig("original_sentiment_pie_chart.png")
plt.close()

# ---------- BALANCING the dataset ----------
balanced_df = final_df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(175000, random_state=42)
).reset_index(drop=True)

# ---------- PIE CHART for BALANCED distribution ----------
balanced_counts = balanced_df["label"].value_counts().sort_index()

plt.figure(figsize=(6, 6))
plt.pie(balanced_counts, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title("Balanced Sentiment Distribution (175k each)")
plt.savefig("balanced_sentiment_pie_chart.png")
plt.close()

print("Pie charts saved as 'original_sentiment_pie_chart.png' and 'balanced_sentiment_pie_chart.png'")


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17000\348008881.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = final_df.groupby('label', group_keys=False).apply(


Pie charts saved as 'original_sentiment_pie_chart.png' and 'balanced_sentiment_pie_chart.png'


In [2]:
print(final_df['label'].value_counts())

label
2    629566
1    371341
0    179050
Name: count, dtype: int64


In [3]:
balanced_df = final_df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(175000, random_state=42)
).reset_index(drop=True)

print(balanced_df['label'].value_counts())
print(balanced_df.head(10))


label
0    175000
1    175000
2    175000
Name: count, dtype: int64
                                                text  label
0  Ordered this two days ago and it's coming toda...      0
1  The Game Ordered to Pay $7 Million After Losin...      0
2  Magic mushrooms help cancer patients cope with...      0
3  Lustima tutorial on YouTube &amp; I still can'...      0
4  Move into this century #FaroeIslands NO LONGER...      0
5  I finally found a female after HOURS of lookin...      0
6  Click here: https://t.co/JNlTQqQNi4Big cock is...      0
7  Why Businesses Can’t Survive Without #SocialMe...      0
8  I don't get the "World's Biggest E-mail" bit i...      0
9  RPBASEU: RT allkpopBuzz: Daycare instructor su...      0


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8064\421661485.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = final_df.groupby('label', group_keys=False).apply(


In [4]:
import re

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)  # remove URLs
    text = re.sub(r'\@\w+|\#','', text)  # remove mentions and hashtags
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove special chars, keep only letters & spaces
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text


In [5]:
balanced_df['clean_text'] = balanced_df['text'].apply(clean_text)

In [7]:
for original, cleaned in zip(balanced_df['text'].head(5), balanced_df['clean_text'].head(5)):
    print("Original:", original)
    print("Cleaned :", cleaned)
    print()


Original: Ordered this two days ago and it's coming today,,,, fuck me up ADFHKLL???????????????? https://t.co/EzLo4oc6rP
Cleaned : ordered this two days ago and its coming today fuck me up adfhkll

Original: The Game Ordered to Pay $7 Million After Losing Sexual Assault Case https://t.co/Ujq7Wv7jtd https://t.co/1YtKLqo42r
Cleaned : the game ordered to pay million after losing sexual assault case

Original: Magic mushrooms help cancer patients cope with fear and depression https://t.co/epVPvqC6wg
Cleaned : magic mushrooms help cancer patients cope with fear and depression

Original: Lustima tutorial on YouTube &amp; I still can't wrap for shit ???????? It's okay #ItsWrappedWithLove https://t.co/ZtOKKTtpDP
Cleaned : lustima tutorial on youtube amp i still cant wrap for shit its okay itswrappedwithlove

Original: Move into this century #FaroeIslands NO LONGER acceptable to kill these  sentient beings.  https://t.co/HsOCVEBJe6 #OpKillingBay #EU
Cleaned : move into this century faroeislands

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Set vocabulary size and max sequence length
vocab_size = 10000
max_len = 100

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(balanced_df['clean_text'])

# Convert texts to sequences
sequences = tokenizer.texts_to_sequences(balanced_df['clean_text'])

# Pad sequences
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')


In [12]:
print("Tokenized sequences (first 5):")
for seq in sequences[:5]:
    print(seq)


Tokenized sequences (first 5):
[3609, 13, 215, 267, 570, 9, 45, 312, 62, 105, 19, 48, 1]
[2, 144, 3609, 3, 988, 619, 99, 986, 2386, 3716, 248]
[1634, 1, 188, 1062, 3610, 5712, 16, 645, 9, 1969]
[1, 3215, 14, 1247, 24, 10, 126, 68, 2445, 8, 145, 45, 747, 1]
[780, 198, 13, 2456, 1, 66, 1414, 8544, 3, 430, 83, 1, 8207, 5889, 1757]


In [13]:
print("Padded sequences shape:", padded_sequences.shape)
print("Padded sequences (first 3):")
print(padded_sequences[:3])


Padded sequences shape: (525000, 100)
Padded sequences (first 3):
[[3609   13  215  267  570    9   45  312   62  105   19   48    1    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]
 [   2  144 3609    3  988  619   99  986 2386 3716  248    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0 

In [14]:
import numpy as np
labels = np.array(balanced_df['label'])

In [15]:
print("Shape of labels:", labels.shape)
print("First 5 labels:", labels[:5])
print("Label distribution:", np.unique(labels, return_counts=True))



Shape of labels: (525000,)
First 5 labels: [0 0 0 0 0]
Label distribution: (array([0, 1, 2]), array([175000, 175000, 175000]))


In [16]:
import numpy as np

np.save("savedPackages/padded_sequences.npy", padded_sequences)
np.save("savedPackages/labels.npy", labels)

In [17]:
import pickle

with open("savedPackages/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)


In [18]:
from sklearn.model_selection import train_test_split

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.2, random_state=42, stratify=labels
)


In [19]:
from tensorflow.keras.layers import Bidirectional

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])


NameError: name 'Sequential' is not defined

In [30]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


In [31]:
# Add EarlyStopping to avoid wasting time
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=512,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/10
821/821 ━━━━━━━━━━━━━━━━━━━━ 862s 1s/step - accuracy: 0.8191 - loss: 0.4290 - val_accuracy: 0.9572 - val_loss: 0.1279
Epoch 2/10
821/821 ━━━━━━━━━━━━━━━━━━━━ 487s 593ms/step - accuracy: 0.9604 - loss: 0.1250 - val_accuracy: 0.9600 - val_loss: 0.1230
Epoch 3/10
821/821 ━━━━━━━━━━━━━━━━━━━━ 477s 581ms/step - accuracy: 0.9649 - loss: 0.1097 - val_accuracy: 0.9604 - val_loss: 0.1193
Epoch 4/10
821/821 ━━━━━━━━━━━━━━━━━━━━ 492s 599ms/step - accuracy: 0.9689 - loss: 0.0979 - val_accuracy: 0.9612 - val_loss: 0.1206
Epoch 5/10
821/821 ━━━━━━━━━━━━━━━━━━━━ 498s 606ms/step - accuracy: 0.9717 - loss: 0.0876 - val_accuracy: 0.9620 - val_loss: 0.1234


In [32]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}, Test Loss: {test_loss:.4f}")


3282/3282 ━━━━━━━━━━━━━━━━━━━━ 73s 22ms/step - accuracy: 0.9605 - loss: 0.1179
Test Accuracy: 0.9604, Test Loss: 0.1193


In [33]:
model.save("sentiment_lstm_model.h5")  # Saves architecture + weights + optimizer state


In [34]:
model.save("sentiment_lstm_model.keras")


In [35]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Predict on test set
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Show classification report
print(classification_report(y_test, y_pred, target_names=["NEG", "POS", "NEU"]))

# Optional: show confusion matrix
print(confusion_matrix(y_test, y_pred))


3282/3282 ━━━━━━━━━━━━━━━━━━━━ 76s 22ms/step
              precision    recall  f1-score   support

         NEG       0.97      0.95      0.96     35000
         POS       0.96      0.96      0.96     35000
         NEU       0.95      0.97      0.96     35000

    accuracy                           0.96    105000
   macro avg       0.96      0.96      0.96    105000
weighted avg       0.96      0.96      0.96    105000

[[33286   666  1048]
 [  579 33666   755]
 [  526   585 33889]]


In [6]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_sentiment(text, model, tokenizer, max_len=100):
    # Preprocess text
    import re
    def clean_text(t):
        t = t.lower()
        t = re.sub(r"http\S+|www\S+|https\S+", '', t)
        t = re.sub(r'\@\w+|\#','', t)
        t = re.sub(r'[^a-zA-Z\s]', '', t)
        t = re.sub(r'\s+', ' ', t).strip()
        return t

    clean = clean_text(text)
    seq = tokenizer.texts_to_sequences([clean])
    padded = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    pred = model.predict(padded)
    label = np.argmax(pred, axis=1)[0]
    label_map = {0: "NEGATIVE", 1: "POSITIVE", 2: "NEUTRAL"}
    return label_map[label]

# Example
predict_sentiment("i am good", model, tokenizer)


NameError: name 'model' is not defined

In [ ]:
["i", "love", "this", "movie"]    [10, 45, 67, 89, 0, 0, 0, 0...]